In [1]:
# Model download + local smoke test (SmolLM2-360M)
# This will download weights the first time you run it (internet required).
# Goal: verify the model runs locally and can generate text.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "HuggingFaceTB/SmolLM2-360M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# Use fp16 on GPU for efficiency; keep fp32 on CPU
dtype = torch.float16 if DEVICE == "cuda" else torch.float32
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype).to(DEVICE)
model.eval()

inputs = tokenizer.encode("Africa is", return_tensors="pt").to(DEVICE)
with torch.no_grad():
    out_ids = model.generate(inputs, max_new_tokens=40)

print("Using device:", DEVICE)
print(tokenizer.decode(out_ids[0], skip_special_tokens=True))

e:\school\BrainInk-Local\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Using device: cpu
Africa is a country that has been in the news a lot lately. It is a country that is in the news because of the ongoing conflict in the country. It is a country that is in the news because


# Reasoning + feedback module (SmolLM2)
This notebook contains the **SmolLM2** side of the workflow: long-form educational feedback generation.

**Contract**
- SmolLM2 never sees raw images.
- It consumes a structured *Student Answer Analysis* text summary produced by SmolDocling in `smol_docling.ipynb`.
- Output must be a **single well-structured paragraph** (no bullet points).

In [3]:
import re
from typing import Optional, Tuple

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def load_smollm2(model_id: str = "HuggingFaceTB/SmolLM2-360M") -> Tuple[AutoTokenizer, AutoModelForCausalLM, str]:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(model_id)
    dtype = torch.float16 if device == "cuda" else torch.float32
    mdl = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype).to(device)
    mdl.eval()
    return tok, mdl, device


def build_feedback_prompt(clip_generated_summary: str) -> str:
    # (Kept template-compatible with the earlier FLAN-T5 workflow.)
    return (
        "You are an experienced mathematics teacher.\n\n"
        "Based on the following student answer analysis, write a detailed feedback paragraph.\n"
        "The feedback must:\n"
        "- Be at least 120 words\n"
        "- Be constructive and pedagogical\n"
        "- Clearly state what the student did well\n"
        "- Clearly explain where the student went wrong\n"
        "- Suggest how the student can improve\n\n"
        "Student Answer Analysis:\n"
        f"{clip_generated_summary.strip()}\n\n"
        "Write the feedback in a formal but supportive academic tone.\n"
        "Output must be a single well-structured paragraph with no bullet points."
    )


def postprocess_single_paragraph(text: str) -> str:
    text = text.strip()
    # remove bullet-like prefixes
    text = re.sub(r"(?m)^\s*[-*•]+\s+", "", text)
    # collapse newlines into a single paragraph
    text = re.sub(r"\s*\n\s*", " ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()


@torch.no_grad()
def generate_feedback(
    student_answer_analysis: str,
    model_id: str = "HuggingFaceTB/SmolLM2-360M",
    max_new_tokens: int = 220,
    min_new_tokens: int = 160,
    temperature: float = 0.7,
    top_p: float = 0.9,
    repetition_penalty: float = 1.15,
    no_repeat_ngram_size: int = 4,
    seed: Optional[int] = 0,
) -> str:
    tok, mdl, device = load_smollm2(model_id=model_id)
    prompt = build_feedback_prompt(student_answer_analysis)

    if seed is not None:
        torch.manual_seed(int(seed))
        if device == "cuda":
            torch.cuda.manual_seed_all(int(seed))

    inputs = tok(prompt, return_tensors="pt").to(device)
    out_ids = mdl.generate(
        **inputs,
        do_sample=True,
        temperature=float(temperature),
        top_p=float(top_p),
        max_new_tokens=int(max_new_tokens),
        min_new_tokens=int(min_new_tokens),
        repetition_penalty=float(repetition_penalty),
        no_repeat_ngram_size=int(no_repeat_ngram_size),
        pad_token_id=tok.eos_token_id,
    )
    text = tok.decode(out_ids[0], skip_special_tokens=True)
    # Keep only the completion (strip the prompt if echoed)
    if text.startswith(prompt):
        text = text[len(prompt):]
    text = postprocess_single_paragraph(text)
    return f"[{model_id} on {device}] {text}"


# Minimal demo summary (you can paste the output summary from smol_docling.ipynb here)
demo_summary = """Image Analysis Summary:
- Subject: Mathematics (Algebra)
- Correctness: Partially correct
- Strengths: Correct setup and mostly logical steps
- Weaknesses: Final simplification error (sign mistake)
- Handwriting: Mostly legible
"""

print(generate_feedback(demo_summary))

[HuggingFaceTB/SmolLM2-360M on cpu] 3. Write a brief response to the next question by emailing your written responses to me (with any relevant attachments). You may use this template for you first draft if it is helpful. What do you think about these questions? In which areas of my course should I revise or rework some content? Should we focus more time/effort into developing our own learning strategies? Can we provide students with better guidance and support when they encounter difficulties? How would you address the issues mentioned above? 2. This will help us understand each other's point of view 4. The purpose of this paper is not just to inform the reader that there has been work done on the topic; rather it aims to contribute something new to the discussion as far as research goes.
